<img src="https://datascientest.fr/train/assets/logo_liora.png" style="height:150px">

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> AI Engineers — Démo Part 10 </h1></center>
<center><h2> Un agent qui cherche et qui lit </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

> **Part 10 · 25 min · ≈ 0,01 $ · prépare le Lab 6 et le chapitre 8 du projet**
> Données : 5 fiches « congés » dans le notebook. Pas d'index, pas de Qdrant. **Sans gateway, la démo se rabat sur les sorties enregistrées.**

```
   question ──> MODÈLE ──┬── "je veux appeler search(...)" ──> PROGRAMME exécute
                  ▲      │                                          │
                  │      │            observation (ToolMessage)     │
                  └──────┼──────────────────────────────────────────┘
                         └── "voici la réponse" ──> FIN
```

In [1]:
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0,
                 base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])

FICHES = {
    "F01": {"title": "Congé pour mariage ou PACS",
            "text": "Le salarié a droit à 4 jours de congé pour son mariage ou son PACS. Ces jours sont payés."},
    "F02": {"title": "Congé pour enfant malade",
            "text": "Le salarié peut s'absenter pour soigner un enfant malade de moins de 16 ans, "
                    "3 jours par an (5 jours si l'enfant a moins d'un an). Un certificat médical est demandé."},
    "F03": {"title": "Congé de proche aidant",
            "text": "Pour aider un proche en perte d'autonomie. Durée maximale : 3 mois, renouvelable, "
                    "dans la limite d'un an sur toute la carrière. Non rémunéré par l'employeur."},
    "F04": {"title": "Congé de solidarité familiale",
            "text": "Pour accompagner un proche en fin de vie. Durée : 3 mois au maximum, renouvelable une fois "
                    "(6 mois au total). Non rémunéré par l'employeur."},
    "F05": {"title": "Congé pour décès d'un proche",
            "text": "Le décès d'un enfant donne droit à 12 jours de congé. Celui du conjoint ou d'un parent, 3 jours."},
}
print(len(FICHES), "fiches :", [f["title"] for f in FICHES.values()])

try:
    llm.invoke("ok")
except Exception as error:
    print("⚠️ gateway inaccessible :", type(error).__name__, "→ les cellules suivantes affichent la sortie enregistrée")

5 fiches : ['Congé pour mariage ou PACS', 'Congé pour enfant malade', 'Congé de proche aidant', 'Congé de solidarité familiale', "Congé pour décès d'un proche"]


<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 1. Deux outils </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Un outil = **une fonction Python** + **une notice** (la docstring).
- Le modèle voit le **nom**, la **docstring** et les **paramètres**, **jamais le code**.

In [2]:
@tool
def search_fiches(query: str) -> str:
    """Search the fiches by keywords. Give a short query about ONE topic. Returns fiche ids and titles."""
    words = [w for w in query.lower().replace("'", " ").split() if len(w) > 3]
    scores = {doc_id: sum(w in (f["title"] + " " + f["text"]).lower() for w in words) for doc_id, f in FICHES.items()}
    best = [doc_id for doc_id, s in sorted(scores.items(), key=lambda x: -x[1]) if s > 0][:3]
    return "\n".join(f"{doc_id} : {FICHES[doc_id]['title']}" for doc_id in best) or "Aucun résultat."


@tool
def read_fiche(doc_id: str) -> str:
    """Read one fiche in full from its id (for example F02)."""
    fiche = FICHES.get(doc_id)
    return f"{fiche['title']}\n{fiche['text']}" if fiche else "Fiche inconnue."


TOOLS = [search_fiches, read_fiche]
for t in TOOLS:
    print(t.name, t.args)
print()
print(search_fiches.invoke({"query": "congé mariage"}))

search_fiches {'query': {'title': 'Query', 'type': 'string'}}
read_fiche {'doc_id': {'title': 'Doc Id', 'type': 'string'}}

F01 : Congé pour mariage ou PACS
F02 : Congé pour enfant malade
F03 : Congé de proche aidant


<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 2. La boucle à la main </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- `bind_tools` : le modèle **connaît** les outils.
- Il **n'exécute rien** : il répond par un `tool_calls` (une **demande**).
- C'est **notre code** qui exécute, puis rend le résultat.

In [3]:
llm_tools = llm.bind_tools(TOOLS)
messages = [SystemMessage("Vous répondez aux questions sur les congés à partir des fiches."),
            HumanMessage("Combien de jours de congé pour mon mariage ?")]

response = None
try:
    response = llm_tools.invoke(messages)
    print("content    :", repr(response.content))
    print("tool_calls :", [(c["name"], c["args"]) for c in response.tool_calls])
except Exception as error:
    response = None
    print("⚠️ gateway inaccessible :", type(error).__name__)
    print("Sortie enregistrée : content '' | tool_calls [('search_fiches', {'query': 'congé mariage'})]")

content    : ''
tool_calls : [('search_fiches', {'query': 'congé mariage'})]


> **Aucune fonction n'a tourné.** Le modèle a seulement écrit : « je voudrais appeler `search_fiches` ».

In [4]:
tools_by_name = {t.name: t for t in TOOLS}

try:
    if response is None:
        raise RuntimeError("no model response")
    messages.append(response)  # the model's request goes back into the conversation

    for step in range(1, 6):
        if not response.tool_calls:
            break
        for call in response.tool_calls:
            print(f"🔧 {call['name']}({call['args']})")
            result = tools_by_name[call["name"]].invoke(call["args"])
            print("📄", result.replace("\n", " | ")[:90])
            messages.append(ToolMessage(content=result, tool_call_id=call["id"]))
        response = llm_tools.invoke(messages)  # the model sees the results and decides what is next
        messages.append(response)

    print("✅", response.content)
    print(f"\n{len(messages)} messages, {step} tour(s) de boucle")
except Exception as error:
    print("⚠️ gateway inaccessible :", type(error).__name__)
    print("Sortie enregistrée : 🔧 search_fiches → 🔧 read_fiche(F01) → ✅ « 4 jours de congé, payés » | 7 messages, 3 tours de boucle")

🔧 search_fiches({'query': 'congé mariage'})
📄 F01 : Congé pour mariage ou PACS | F02 : Congé pour enfant malade | F03 : Congé de proche 


🔧 read_fiche({'doc_id': 'F01'})
📄 Congé pour mariage ou PACS | Le salarié a droit à 4 jours de congé pour son mariage ou son


✅ Pour votre mariage ou PACS, vous avez droit à 4 jours de congé, qui sont payés.

7 messages, 3 tour(s) de boucle


- Le **`for`** est la boucle : on s'arrête quand le modèle **ne demande plus d'outil**.
- Il faut gérer : l'**historique**, la **limite de tours**, les **erreurs**. → `create_agent` le fait pour nous.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 3. Le même avec create_agent </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

In [5]:
from langchain.agents import create_agent
from langgraph.errors import GraphRecursionError

PROMPT = """Vous répondez aux questions de salariés sur leurs congés, à partir des fiches.

Règles :
- Vous devez appeler search_fiches avant de répondre. Une recherche par sujet.
- Répondez uniquement à partir des fiches lues, en français, de façon concise.
- Si les fiches ne permettent pas de répondre avec certitude, répondez exactement : NO_ANSWER."""

agent = create_agent(llm, TOOLS, system_prompt=PROMPT)

graph = agent.get_graph()
for edge in graph.edges:
    print(f"{edge.source:>10} {'╌╌>' if edge.conditional else '───>'} {edge.target}")

 __start__ ───> model
     model ╌╌> __end__
     model ╌╌> tools
     tools ╌╌> model


```
 __start__ ───> model ╌╌> tools ───> model ╌╌> __end__
                  ▲_________________|
                       (╌╌> = le modèle choisit)
```
- Un **graphe LangGraph** à deux nœuds : `model` et `tools`.
- La **boucle `for` d'avant** est devenue **une arête qui revient**.

In [6]:
def run(question, agent=agent, limit=25, verbose=True, quiet=False, recorded=None):
    """Run the agent and print its steps one per line. Returns (tool calls, last answer, searches).

    If the gateway is unreachable and `recorded` = (tool calls, searches, answer) is given, it is used instead.
    """
    say = (lambda *args, **kwargs: None) if quiet else print
    say("❓", question)
    n_calls, n_searches, total_tokens, answer = 0, 0, 0, ""
    seen = 1  # the first message is the question itself
    try:
        for state in agent.stream({"messages": [("user", question)]}, config={"recursion_limit": limit},
                                  stream_mode="values"):
            for m in state["messages"][seen:]:
                if m.type == "ai" and m.tool_calls:
                    for c in m.tool_calls:
                        n_calls += 1
                        n_searches += c["name"] == "search_fiches"
                        say(f"🔧 {c['name']}({c['args']})")
                elif m.type == "tool":
                    say("📄", m.content.replace("\n", " | ")[:90])
                elif m.type == "ai":
                    answer = m.content
                    say("✅", answer)
                if m.type == "ai":
                    total_tokens += m.usage_metadata["input_tokens"]
                    if verbose:
                        say(f"     ↳ {m.usage_metadata['input_tokens']} tokens en entrée")
            seen = len(state["messages"])
    except GraphRecursionError:
        say(f"🛑 GraphRecursionError : plafond de {limit} pas atteint")
        answer = "STOPPED"
    except Exception as error:
        print("⚠️ gateway inaccessible :", type(error).__name__)
        if recorded is None:
            raise
        n_calls, n_searches, answer = recorded
        print(f"Sortie enregistrée : {n_calls} appel(s) d'outil | {answer}")
        total_tokens = None
    if total_tokens is not None:
        say(f"→ {n_calls} appel(s) d'outil, {total_tokens} tokens d'entrée au total\n")
    return n_calls, answer, n_searches


_ = run("Combien de jours de congé pour mon mariage ?",
        recorded=(2, 1, "Vous avez droit à 4 jours de congé pour votre mariage ou votre PACS, et ces jours sont payés."))

❓ Combien de jours de congé pour mon mariage ?


🔧 search_fiches({'query': 'congé mariage'})
     ↳ 182 tokens en entrée
📄 F01 : Congé pour mariage ou PACS | F02 : Congé pour enfant malade | F03 : Congé de proche 


🔧 read_fiche({'doc_id': 'F01'})
     ↳ 237 tokens en entrée
📄 Congé pour mariage ou PACS | Le salarié a droit à 4 jours de congé pour son mariage ou son


✅ Vous avez droit à 4 jours de congé pour votre mariage ou votre PACS, et ces jours sont payés.
     ↳ 296 tokens en entrée
→ 2 appel(s) d'outil, 715 tokens d'entrée au total



- **Même déroulé** qu'à la main : 🔧 → 📄 → ✅.
- Les **tokens d'entrée grossissent à chaque tour** : l'historique est **renvoyé en entier**.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 4. Trois questions, trois comportements </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- **Deux sujets** → on attend **deux recherches**.
- **Hors corpus** → on attend un **refus**.
- C'est **le modèle** qui décide, pas notre code.

In [7]:
REFUSALS = ("NO_ANSWER", "ne peux pas aider")


def verdict(n_calls, answer, n_searches, in_corpus):
    """Print an explicit verdict on how the agent behaved (searches, reads, right or wrong refusal)."""
    refused = any(text in answer for text in REFUSALS)
    print(f"   {n_searches} recherche(s), {n_calls - n_searches} lecture(s)")
    if n_searches == 0:
        print("⚠️  Aucune recherche : la réponse n'est appuyée sur aucune fiche.")
    if refused and in_corpus:
        print("❌ refus à tort")
    elif refused:
        print("✅ refus juste")
    elif in_corpus:
        print("✅ réponse donnée")
    else:
        print("❌ réponse donnée à une question hors corpus")


print("Deux sujets :")
verdict(*run("Quelle est la durée du congé de proche aidant et du congé de solidarité familiale ?", verbose=False,
             recorded=(4, 2, "Proche aidant : 3 mois maximum, renouvelable, dans la limite d'un an. Solidarité familiale : 3 mois au maximum, renouvelable une fois (6 mois au total).")),
        in_corpus=True)
print("Hors corpus :")
verdict(*run("Quel est le SMIC ?", verbose=False, recorded=(1, 1, "NO_ANSWER")), in_corpus=False)

Deux sujets :
❓ Quelle est la durée du congé de proche aidant et du congé de solidarité familiale ?


🔧 search_fiches({'query': 'congé de proche aidant'})
🔧 search_fiches({'query': 'congé de solidarité familiale'})
📄 F03 : Congé de proche aidant | F04 : Congé de solidarité familiale | F05 : Congé pour décè
📄 F04 : Congé de solidarité familiale | F01 : Congé pour mariage ou PACS | F02 : Congé pour 


🔧 read_fiche({'doc_id': 'F03'})
🔧 read_fiche({'doc_id': 'F04'})
📄 Congé de proche aidant | Pour aider un proche en perte d'autonomie. Durée maximale : 3 moi
📄 Congé de solidarité familiale | Pour accompagner un proche en fin de vie. Durée : 3 mois a


✅ Le congé de proche aidant a une durée maximale de 3 mois, renouvelable, dans la limite d'un an sur toute la carrière. Le congé de solidarité familiale dure également 3 mois au maximum, renouvelable une fois, pour un total de 6 mois. Les deux congés ne sont pas rémunérés par l'employeur.
→ 4 appel(s) d'outil, 976 tokens d'entrée au total

   2 recherche(s), 2 lecture(s)
✅ réponse donnée
Hors corpus :
❓ Quel est le SMIC ?


🔧 search_fiches({'query': 'SMIC'})
📄 Aucun résultat.


✅ NO_ANSWER
→ 1 appel(s) d'outil, 383 tokens d'entrée au total

   1 recherche(s), 0 lecture(s)
✅ refus juste


<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 5. L'agent qui répond sans chercher </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Observé au projet : **un refus à tort, sans aucun appel d'outil**.
- Ici : une règle de **filtre** dans le prompt (« hors congés → refusez ») **et** « cherchez avant de répondre ».
- Question : « Mon fils est malade, combien de jours ? » (le mot « congé » n'y est pas)

In [8]:
GATE_PROMPT = """Vous répondez aux questions sur les congés des salariés. Si la question ne concerne pas les congés, répondez que vous ne pouvez pas aider. Cherchez avant de répondre."""

QUESTION = "Mon fils est malade, combien de jours ?"
gate_agent = create_agent(llm, TOOLS, system_prompt=GATE_PROMPT)
verdict(*run(QUESTION, agent=gate_agent, verbose=False, recorded=(0, 0, "Je ne peux pas aider avec cette question.")), in_corpus=True)
verdict(*run("Combien de jours de congé pour mon mariage ?", agent=gate_agent, verbose=False,
             recorded=(2, 1, "Vous avez droit à 4 jours de congé pour votre mariage ou votre PACS, et ces jours sont payés.")), in_corpus=True)

❓ Mon fils est malade, combien de jours ?


✅ Je ne peux pas aider avec cette question.
→ 0 appel(s) d'outil, 137 tokens d'entrée au total

   0 recherche(s), 0 lecture(s)
⚠️  Aucune recherche : la réponse n'est appuyée sur aucune fiche.
❌ refus à tort
❓ Combien de jours de congé pour mon mariage ?


🔧 search_fiches({'query': 'congé mariage'})
📄 F01 : Congé pour mariage ou PACS | F02 : Congé pour enfant malade | F03 : Congé de proche 


🔧 read_fiche({'doc_id': 'F01'})
📄 Congé pour mariage ou PACS | Le salarié a droit à 4 jours de congé pour son mariage ou son


✅ Vous avez droit à 4 jours de congé pour votre mariage ou votre PACS, et ces jours sont payés.
→ 2 appel(s) d'outil, 586 tokens d'entrée au total

   1 recherche(s), 1 lecture(s)
✅ réponse donnée


In [9]:
REPLAYS = 5
try:
    silent_refusals = sum(run(QUESTION, agent=gate_agent, quiet=True)[0] == 0 for _ in range(REPLAYS))
    mark = "❌" if silent_refusals else "✅"
    print(f"{mark} {silent_refusals} refus sans recherche sur {REPLAYS} rejeux de la même question")
    print("(en direct, vos valeurs varient ; au chapitre 8 : 3 refus à tort sur 5 rejoués, sans aucun appel d'outil)")
except Exception:
    print("Sortie enregistrée : refus sans recherche à l'enregistrement (0 appel d'outil) ; chapitre 8 : 3 refus à tort sur 5 rejoués, sans appel d'outil.")

❌ 5 refus sans recherche sur 5 rejeux de la même question
(en direct, vos valeurs varient ; au chapitre 8 : 3 refus à tort sur 5 rejoués, sans aucun appel d'outil)


> **Refus à tort, aucune erreur.** « Cherchez avant de répondre » était une **suggestion** : le modèle a jugé la question hors sujet **sans chercher**.

In [10]:
FIXED_PROMPT = """Vous répondez aux questions sur les congés des salariés. Vous devez appeler search_fiches avant de répondre. Si les fiches ne permettent pas de répondre, dites que vous ne pouvez pas aider."""

fixed_agent = create_agent(llm, TOOLS, system_prompt=FIXED_PROMPT)
verdict(*run(QUESTION, agent=fixed_agent, verbose=False,
             recorded=(2, 1, "Un salarié peut s'absenter pour soigner un enfant malade de moins de 16 ans pendant 3 jours par an (5 jours si l'enfant a moins d'un an).")),
        in_corpus=True)
print("Hors corpus, avec le même prompt :")
verdict(*run("Quel est le SMIC ?", agent=fixed_agent, verbose=False, recorded=(1, 1, "Je ne peux pas aider avec cette question.")), in_corpus=False)

❓ Mon fils est malade, combien de jours ?


🔧 search_fiches({'query': 'congés maladie enfant'})
📄 F02 : Congé pour enfant malade | F05 : Congé pour décès d'un proche


🔧 read_fiche({'doc_id': 'F02'})
📄 Congé pour enfant malade | Le salarié peut s'absenter pour soigner un enfant malade de moi


✅ Un salarié peut s'absenter pour soigner un enfant malade de moins de 16 ans pendant 3 jours par an. Si l'enfant a moins d'un an, le congé peut aller jusqu'à 5 jours. Un certificat médical est requis.
→ 2 appel(s) d'outil, 586 tokens d'entrée au total

   1 recherche(s), 1 lecture(s)
✅ réponse donnée
Hors corpus, avec le même prompt :
❓ Quel est le SMIC ?


🔧 search_fiches({'query': 'SMIC'})
📄 Aucun résultat.


✅ Je ne peux pas aider avec cette question.
→ 1 appel(s) d'outil, 301 tokens d'entrée au total

   1 recherche(s), 0 lecture(s)
✅ refus juste


- **« Vous devez appeler »**, sans filtre : l'agent **cherche**, et le hors-sujet est refusé **après** la recherche.
- Le comportement **varie selon la formulation** et **d'une exécution à l'autre** : c'est ce que le chapitre 8 **mesure** (zéro appel d'outil = refus à tort). **Au chapitre 8, le SMIC était refusé sans recherche** ; avec ce prompt-ci, il peut l'être après une recherche.
- Mais **trop insister** (« cherchez jusqu'à trouver ») fait **boucler** l'agent → section suivante.
- Ce défaut ne se voit qu'en **regardant les appels d'outils**.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 6. Le garde-fou et le coût </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Un prompt trop insistant + une recherche **sans résultat** = **une boucle** (ici **simulée** : un outil qui ne trouve jamais ; cas réel : chapitre 8, section 8.7).
- Chaque tour **renvoie tout l'historique** : le coût **croît plus vite que le nombre de tours**.
- Garde-fou : **`recursion_limit`**.

In [11]:
@tool("search_fiches")
def empty_search(query: str) -> str:
    """Search the fiches by keywords."""
    return "Aucun résultat. Reformulez la requête."


STUBBORN_PROMPT = """Vous répondez aux questions sur les congés.
Vous devez trouver la réponse dans les fiches. Tant que la recherche ne donne rien, reformulez et recommencez. Ne renoncez jamais."""

stubborn_agent = create_agent(llm, [empty_search], system_prompt=STUBBORN_PROMPT)
_ = run("Combien de jours de congé pour mon mariage ?", agent=stubborn_agent, limit=11, recorded=(6, 6, "STOPPED"))

❓ Combien de jours de congé pour mon mariage ?


🔧 search_fiches({'query': 'jours de congé mariage'})
     ↳ 99 tokens en entrée
📄 Aucun résultat. Reformulez la requête.


🔧 search_fiches({'query': 'congés pour mariage'})
     ↳ 138 tokens en entrée
📄 Aucun résultat. Reformulez la requête.


🔧 search_fiches({'query': 'jours de congés pour un mariage'})
     ↳ 177 tokens en entrée
📄 Aucun résultat. Reformulez la requête.


🔧 search_fiches({'query': 'congés mariage'})
     ↳ 218 tokens en entrée
📄 Aucun résultat. Reformulez la requête.


🔧 search_fiches({'query': 'congés exceptionnels mariage'})
     ↳ 256 tokens en entrée
📄 Aucun résultat. Reformulez la requête.


🔧 search_fiches({'query': 'jours de congé pour un mariage'})
     ↳ 296 tokens en entrée
🛑 GraphRecursionError : plafond de 11 pas atteint
→ 6 appel(s) d'outil, 1184 tokens d'entrée au total



- Chaque appel du modèle ou d'un outil est **un pas** : au **11e**, `GraphRecursionError`.
- Les tokens d'entrée **montent à chaque tour** (historique renvoyé) : le **coût** suit.
- Sans plafond : **la boucle tourne jusqu'à la facture**.
- Au projet : on **attrape l'erreur** et on rend un **refus**.

In [12]:
REFUSAL_TEXT = "Je n'ai pas pu répondre, la recherche n'a rien donné."


def ask_safely(agent, question, limit):
    """Answer the question, or return a clean refusal if the step limit is reached."""
    try:
        return agent.invoke({"messages": [("user", question)]}, config={"recursion_limit": limit})["messages"][-1].content
    except GraphRecursionError as error:
        print("🛑", type(error).__name__, "attrapée")
        return REFUSAL_TEXT


try:
    print("Réponse rendue à l'utilisateur :", ask_safely(stubborn_agent, "Combien de jours pour un mariage ?", limit=7))
except Exception as error:
    print("⚠️ gateway inaccessible :", type(error).__name__)
    print("Sortie enregistrée : 🛑 GraphRecursionError attrapée → « " + REFUSAL_TEXT + " »")

🛑 GraphRecursionError attrapée
Réponse rendue à l'utilisateur : Je n'ai pas pu répondre, la recherche n'a rien donné.


<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 7. Ce que create_agent construit </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Le **même agent** avec un `StateGraph` : **2 nœuds** et **1 aiguillage**.
- `tools_condition` : des `tool_calls` → `tools`, sinon → fin.

In [13]:
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

def call_model(state: MessagesState):
    """The model node: one model call on the whole history."""
    return {"messages": [llm.bind_tools(TOOLS).invoke([SystemMessage(FIXED_PROMPT)] + state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)  # tool_calls -> "tools", else -> END
builder.add_edge("tools", "model")                       # the loop
my_agent = builder.compile()

verdict(*run("Combien de jours de congé pour un enfant malade ?", agent=my_agent, verbose=False,
             recorded=(2, 1, "Un salarié peut s'absenter pour soigner un enfant malade de moins de 16 ans pendant 3 jours par an (5 jours si l'enfant a moins d'un an).")),
        in_corpus=True)

❓ Combien de jours de congé pour un enfant malade ?


🔧 search_fiches({'query': 'congé enfant malade'})
📄 F02 : Congé pour enfant malade | F05 : Congé pour décès d'un proche | F01 : Congé pour mar


🔧 read_fiche({'doc_id': 'F02'})
📄 Congé pour enfant malade | Le salarié peut s'absenter pour soigner un enfant malade de moi


✅ Un salarié peut s'absenter pour soigner un enfant malade de moins de 16 ans pendant 3 jours par an. Si l'enfant a moins d'un an, le congé est de 5 jours par an. Un certificat médical est requis.
→ 2 appel(s) d'outil, 615 tokens d'entrée au total

   1 recherche(s), 1 lecture(s)
✅ réponse donnée


- **Une quinzaine de lignes** : même déroulé, mêmes outils.
- `create_agent` ajoute : sortie structurée, middlewares, mémoire. **Le principe reste cette boucle.**

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > Récapitulatif </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

| Ce qu'on a vu | Verdict | Où vous le mettez en pratique |
|---|---|---|
| **Le modèle demande, le programme exécute** (`tool_calls` → `ToolMessage`) | le principe de tout agent | **Lab 6** |
| **`create_agent`** : la boucle dans un graphe LangGraph | à utiliser, comme au projet | **Projet, chapitre 8** |
| **Un prompt est une suggestion** : l'agent peut répondre sans chercher | indétectable sans regarder les appels | **Projet, chapitre 8** (section 8.6) |
| **Trop insister** fait boucler | `recursion_limit` + `GraphRecursionError` attrapée | **Projet, chapitre 8** (section 8.7) |
| **Le coût croît à chaque tour** (historique renvoyé) | plafonner, mesurer | **Part 12** |
| **`StateGraph`** : ce que `create_agent` construit | pour les flux sur mesure | **Aller plus loin** (pas de lab dédié) |

- **Retenez** : un agent = **une boucle**, et **un modèle qui décide**.
- **Mesurer le comportement** (appels d'outils), pas seulement la réponse.
- **Prochain pas : le Lab 6.**